In [ ]:
#include <Wire.h>
#include <RTClib.h>

RTC_DS3231 rtc;

#define MAX_MEMORY 1024
#define ENTRY_SIZE 8

// -------- GLOBAL VARIABLES --------
int R1 = 9620;   
int R2 = 9650;   
float DIVIDER_FACTOR = (float)(R1 + R2) / R2;

uint16_t currentAddress;
unsigned long lastLogTime = 0;
const unsigned long ONE_HOUR_MS = 1000; // Currently set to 1s for testing

#define SENSOR_PIN 26

bool memoryFull = false;
bool uploadMode = false;

// =====================================================
// SENSOR FUNCTION
// =====================================================
int readTurbiditySensor() {
  int rawValue = analogRead(SENSOR_PIN); 
  float v_raw = rawValue * (3.3 / 4095.0); 
  float v_val = v_raw * DIVIDER_FACTOR;

  // Polynomial formula
  float T_value = (496.8 * sq(v_val)) - (3804.8 * v_val) + 7154.5;

  // Constrain T_value to realistic sensor limits (0-4000 NTU typical)
  if (T_value < 0) T_value = 0.0;
  if (T_value > 4000) T_value = 4000.0; 

  return (int)(T_value * 10.0);
}

// =====================================================
// EEPROM HELPERS
// =====================================================
void writeEEPROM(uint16_t addr, byte data) {
  byte block = addr / 256;
  byte memAddr = addr % 256;
  byte deviceAddr = 0x50 + block;

  Wire.beginTransmission(deviceAddr);
  Wire.write(memAddr);
  Wire.write(data);
  Wire.endTransmission();
  delay(5);
}

byte readEEPROM(uint16_t addr) {
  byte block = addr / 256;
  byte memAddr = addr % 256;
  byte deviceAddr = 0x50 + block;

  Wire.beginTransmission(deviceAddr);
  Wire.write(memAddr);
  Wire.endTransmission();
  Wire.requestFrom(deviceAddr, 1);
  return Wire.available() ? Wire.read() : 0xFF;
}

void updateIndex(uint16_t newAddr) {
  writeEEPROM(0, newAddr >> 8);
  writeEEPROM(1, newAddr & 0xFF);
  currentAddress = newAddr;
}

void dumpData() {
  Serial.println("UPLOADING DATA...");
  Serial.println("--- LOG START ---");
  for (uint16_t i = 2; i < currentAddress; i += ENTRY_SIZE) {
    byte mo = readEEPROM(i);
    if (mo == 255) continue; // Skip empty slots

    Serial.print(mo); Serial.print("/");
    Serial.print(readEEPROM(i+1)); Serial.print("/");
    Serial.print(readEEPROM(i+2)); Serial.print(" ");
    Serial.print(readEEPROM(i+3)); Serial.print(":");
    Serial.print(readEEPROM(i+4)); Serial.print(":");
    Serial.print(readEEPROM(i+5)); Serial.print(" -> ");

    int turbScaled = (readEEPROM(i+6) << 8) | readEEPROM(i+7);
    Serial.println(turbScaled / 10.0, 1);
  }
  Serial.println("--- END_OF_DATA ---");
}

// =====================================================
// SETUP
// =====================================================
void setup() {
  Serial.begin(9600);
  delay(1000);
  Wire.begin();
  analogReadResolution(12);

  if (!rtc.begin()) {
    Serial.println("Couldn't find RTC");
  }

  // FIX: If RTC lost power, set the time to the sketch compile time
  if (rtc.lostPower()) {
    Serial.println("RTC lost power, setting time...");
   
  }

  uint16_t high = readEEPROM(0);
  uint16_t low  = readEEPROM(1);
  currentAddress = (high << 8) | low;

  if (currentAddress < 2 || currentAddress >= MAX_MEMORY) {
    currentAddress = 2;
    updateIndex(currentAddress);
  }
}

// =====================================================
// LOOP
// =====================================================
void loop() {
  if (Serial.available()) {
    String cmd = Serial.readStringUntil('\n');
    cmd.trim();
    if (cmd == "upload the data") {
      uploadMode = true;
      dumpData();
      uploadMode = false;
    }
  }

  if (uploadMode || memoryFull) return;

  unsigned long currentTime = millis();
  if (currentTime - lastLogTime >= ONE_HOUR_MS) {
    if (currentAddress + ENTRY_SIZE > MAX_MEMORY) {
      memoryFull = true;
      return;
    }

    DateTime now = rtc.now();
    int turbidityScaled = readTurbiditySensor();

    // PRINTING TO SERIAL DURING LOGGING (Requested)
    Serial.print("Logged @ ");
    Serial.print(now.hour()); Serial.print(":"); Serial.print(now.minute());
    Serial.print(" | Turbidity: ");
    Serial.println(turbidityScaled / 10.0, 1);

    uint16_t addr = currentAddress;
    writeEEPROM(addr++, now.month());
    writeEEPROM(addr++, now.day());
    writeEEPROM(addr++, now.year() - 2000);
    writeEEPROM(addr++, now.hour());
    writeEEPROM(addr++, now.minute());
    writeEEPROM(addr++, now.second());
    writeEEPROM(addr++, turbidityScaled >> 8);
    writeEEPROM(addr++, turbidityScaled & 0xFF);

    updateIndex(addr);
    lastLogTime = currentTime;
  }
}